# 00 环境与数据检查
先运行环境检查，再选择数据路径。此 Notebook 不训练模型；默认只读前 20 张图片，不代表完成全数据核验。

In [ ]:
import json
import sys
from pathlib import Path

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'sop_project' / 'data.py').is_file()), None)
if root is None:
    raise RuntimeError('请从克隆的仓库目录或其 notebooks 目录启动 Notebook。')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from sop_project.environment import check_environment
print(json.dumps(check_environment(), ensure_ascii=False, indent=2))

只有提示缺少 Pillow/IPython 等基础依赖时才运行下一格，安装到当前 Notebook 内核。它不会安装或替换 PyTorch；PyTorch 版本等确认显卡后再定。

In [ ]:
import subprocess

# 需要安装基础依赖时，再将 False 改为 True。
INSTALL_BASE = False
if INSTALL_BASE:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(root / 'requirements-base.txt')])

In [ ]:
from sop_project.data import SOPDataset, configured_path

try:
    data_path = configured_path()
except ValueError:
    data_path = input('输入 SOP ZIP 或已解压数据文件夹的完整路径：').strip()

with SOPDataset(data_path) as dataset:
    images = dataset.load_index()
    report, eligible_preview = dataset.audit(images, limit=20)

print(json.dumps(report, ensure_ascii=False, indent=2))
print('默认仅核验前 20 张图片；不要用 eligible_preview 创建正式实验数据集。')

In [ ]:
from IPython.display import display

with SOPDataset(data_path) as dataset:
    picture = dataset.read_image(images[0])
print('第一张图片尺寸：', picture.size)
display(picture)
# 提交到 GitHub 前清除所有输出，避免将数据集图片嵌入 Notebook。

完成后提交环境检查与核验统计输出给组内确认。完整扫描可在终端使用 `python -m sop_project.data --full`，需要先配置本地路径。商品划分、配对、特征提取及模型不在本阶段实现。